In [1]:
partition = 478

In [2]:
import sys
from train import main
from itertools import product  
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


In [3]:
import re

def load_tested_configs(log_path):
    tested = set()
    with open(log_path, 'r') as f:
        for line in f:
            if line.startswith("Running:"):
                match = re.findall(r"[-\w.]+=\S+", line)
                if match:
                    # Normalize values to correct types
                    config = tuple([
                        int(re.search(r"=(\d+)", match[0]).group(1)),       # n_tree
                        int(re.search(r"=(\d+)", match[1]).group(1)),       # t_depth
                        int(re.search(r"=(\d+)", match[2]).group(1)),       # hd
                        int(re.search(r"=(\d+)", match[3]).group(1)),       # batch_size
                        float(re.search(r"=(\d+\.?\d*)", match[4]).group(1)), # feature_rate
                        float(re.search(r"=(\d+\.?\d*)", match[5]).group(1)), # dropout
                        float(re.search(r"=(\d+\.?\d*)", match[6]).group(1)), # lr
                    ])
                    tested.add(config)
    return tested


In [4]:
import random
from itertools import product
import sys

log_path = f"logs{partition}.txt"
#tested_configs = load_tested_configs(log_path)

n_tree_values = [5, 10, 20, 50, 100]
tree_depth_values = [8, 9, 10, 11,12,13]
hidden_dim = [1024, 768]
batch_size_values = [128, 256, 512]
tree_feature_rates = [0.0, 0.1, 0.2, 0.3, 0.4]
feat_dropouts = [0.0, 0.1, 0.2, 0.3]
lrs = [0.001, 0.01]

n_iter = 50
best_score = 0
best_config = {}

param_space = list(product(
    n_tree_values,
    tree_depth_values,
    hidden_dim,
    batch_size_values,
    tree_feature_rates,
    feat_dropouts,
    lrs
))

best_acc = 0

#available_configs = [cfg for cfg in param_space if cfg not in tested_configs]
available_configs = [cfg for cfg in param_space]
sampled_configs = random.sample(available_configs, min(n_iter, len(available_configs)))
i = 1

for n_tree, t_depth, hd, batch_size, feature_rate, dropout, lr in sampled_configs:
    log_line = f"Running: n_tree={n_tree}, t_depth={t_depth}, hd={hd}, batch_size={batch_size}, feature_rate={feature_rate}, dropout={dropout}, lr={lr}"
    print(f"\n{log_line}")
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{log_line}\n")
    
    sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(n_tree),
        '-tree_depth', str(t_depth),
        '-batch_size', str(batch_size),
        '-hidden_dim', str(hd),
        '-tree_feature_rate', str(feature_rate),
        '-feat_dropout', str(dropout),
        '-lr', str(lr),
        '-epochs', '400',
        '-verbose', '0',
        '-jointly_training',
        '-searching', '1'
    ]

    print(f"{i} / 100")
    acc = main()
    #print(acc)
    with open(log_path, "a") as log_file:
        log_file.write(f"\n{acc}\n")
    i =i + 1

    if acc > best_acc:
        best_acc = acc
        best_config = {
            'n_tree': n_tree,
            'tree_depth': t_depth,
            'batch_size': batch_size,
            'hidden_dim': hd,
            'tree_feature_rate': feature_rate,
            'feat_dropout': dropout,
            'lr': lr
        }

print("\nBest hyperparameter configuration:")
print(best_config)
print(f"Best accuracy: {best_acc}")



Running: n_tree=50, t_depth=11, hd=1024, batch_size=128, feature_rate=0.1, dropout=0.2, lr=0.01
1 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [34:42<00:00,  5.21s/it]
/opt/conda/lib/python3.11/site-packages/torch/nn/init.py:452: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")



Best Accuracy: 0.591317

Running: n_tree=20, t_depth=10, hd=768, batch_size=256, feature_rate=0.0, dropout=0.3, lr=0.01
2 / 100
Use gtd478 dataset
Patience: 100


Training Epochs:  25%|██▌       | 100/400 [01:42<05:08,  1.03s/it]

Early stopping at epoch 101

Best Accuracy: 0.033433

Running: n_tree=100, t_depth=12, hd=768, batch_size=256, feature_rate=0.0, dropout=0.2, lr=0.01
3 / 100
Use gtd478 dataset


...


Best Accuracy: 0.561876

Running: n_tree=5, t_depth=11, hd=1024, batch_size=512, feature_rate=0.2, dropout=0.3, lr=0.001
49 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [01:23<00:00,  4.79it/s]



Best Accuracy: 0.513473

Running: n_tree=5, t_depth=11, hd=768, batch_size=128, feature_rate=0.2, dropout=0.3, lr=0.001
50 / 100
Use gtd478 dataset
Patience: 100


Training Epochs: 100%|██████████| 400/400 [04:12<00:00,  1.58it/s]


Best Accuracy: 0.554391

Best hyperparameter configuration:
{'n_tree': 50, 'tree_depth': 11, 'batch_size': 128, 'hidden_dim': 768, 'tree_feature_rate': 0.1, 'feat_dropout': 0.1, 'lr': 0.01}
Best accuracy: 0.6032934131736527


In [5]:
#Running: n_tree=100, t_depth=11, hd=768, batch_size=512, feature_rate=0.3, dropout=0.1, lr=0.01
# 0.9067

# {'n_tree': 100, 'tree_depth': 11, 'batch_size': 512, 'hidden_dim': 768, 'tree_feature_rate': 0.3, 'feat_dropout': 0.1, 'lr': 0.01}
#0.9174


In [6]:
"""


========== Final Test Evaluation ==========
Model Parameters:
  Dataset: gtd478
  Hidden Dim: 768
  n_tree: 50, tree_depth: 13, tree_feature_rate: 0.1
  Batch size: 512, Dropout: 0.2, LR: 0.01

Best Accuracy: 0.9289
Weighted Precision: 0.9294, Recall: 0.9289, F1 Score: 0.9288, ROCAUC: 0.9976
Macro Precision: 0.9294, Recall: 0.9289, F1 Score: 0.9288, ROCAUC: 0.9976
Micro Precision: 0.9289, Recall: 0.9289, F1 Score: 0.9289, ROCAUC: 0.9985

"""

'\n\n\n========== Final Test Evaluation ==========\nModel Parameters:\n  Dataset: gtd478\n  Hidden Dim: 768\n  n_tree: 50, tree_depth: 13, tree_feature_rate: 0.1\n  Batch size: 512, Dropout: 0.2, LR: 0.01\n\nBest Accuracy: 0.9289\nWeighted Precision: 0.9294, Recall: 0.9289, F1 Score: 0.9288, ROCAUC: 0.9976\nMacro Precision: 0.9294, Recall: 0.9289, F1 Score: 0.9288, ROCAUC: 0.9976\nMicro Precision: 0.9289, Recall: 0.9289, F1 Score: 0.9289, ROCAUC: 0.9985\n\n'

In [7]:
sys.argv = [
        'train.py',
        '-dataset', f'gtd{partition}',
        '-n_class', '30',
        '-gpuid', '0',
        '-n_tree', str(best_config['n_tree']),
        '-tree_depth', str(best_config['tree_depth']),
        '-batch_size', str(best_config['batch_size']),
        '-hidden_dim', str(best_config['hidden_dim']),
        '-epochs', '1500',
        '-verbose', '0',
        '-tree_feature_rate', str(best_config['tree_feature_rate']),
        '-feat_dropout', str(best_config['feat_dropout']),
        '-lr', str(best_config['lr']),
        '-jointly_training',
        '-searching', '0'
    ]

best_model, preds, targets, labels, epoch_logs = main()

Use gtd478 dataset
Patience: 300


Training Epochs:   3%|▎         | 50/1500 [04:13<2:00:14,  4.98s/it]

[Epoch 50] Train Loss: 1.3455, Eval Loss: 1.6286, Eval Accuracy: 0.5474


Training Epochs:   7%|▋         | 100/1500 [08:23<1:57:30,  5.04s/it]

[Epoch 100] Train Loss: 1.2761, Eval Loss: 1.6238, Eval Accuracy: 0.5614


...

[Epoch 1400] Train Loss: 1.1571, Eval Loss: 1.8011, Eval Accuracy: 0.6148


Training Epochs:  97%|█████████▋| 1450/1500 [2:01:22<04:10,  5.00s/it]

[Epoch 1450] Train Loss: 1.1516, Eval Loss: 1.7900, Eval Accuracy: 0.6218


Training Epochs: 100%|██████████| 1500/1500 [2:05:34<00:00,  5.02s/it]

[Epoch 1500] Train Loss: 1.1505, Eval Loss: 1.7793, Eval Accuracy: 0.6193
Evaluating on test set with best model...


In [8]:
from sklearn.metrics import classification_report

print(classification_report(targets, preds))

                                                  precision    recall  f1-score   support

                          Abu Sayyaf Group (ASG)       0.26      0.47      0.34       144
        African National Congress (South Africa)       0.49      0.88      0.63       144
                                Al-Qaida in Iraq       0.40      0.62      0.49       144
        Al-Qaida in the Arabian Peninsula (AQAP)       0.49      0.38      0.42       144
                                      Al-Shabaab       0.20      0.15      0.17       144
             Basque Fatherland and Freedom (ETA)       0.55      0.80      0.65       144
                                      Boko Haram       0.37      0.31      0.34       144
  Communist Party of India - Maoist (CPI-Maoist)       0.57      0.67      0.62       144
       Corsican National Liberation Front (FLNC)       0.69      0.74      0.71       144
                       Donetsk People's Republic       0.49      0.72      0.58       144
Farabundo

In [9]:
def plot_confusion_matrix(y_true, y_pred, labels, partition):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)))
    cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(18, 16))
    sns.heatmap(cm_normalized,
                annot=True,
                fmt=".2f",
                xticklabels=labels,
                yticklabels=labels,
                cmap="viridis",
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})

    plt.title(f"Normalized Confusion Matrix (Partition gtd{partition})", fontsize=18)
    plt.xlabel("Predicted Label", fontsize=14)
    plt.ylabel("True Label", fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()

    save_path = f"results/confusion_matrix_partition_gtd{partition}.png"
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved confusion matrix for partition gtd{partition} to {save_path}")



In [10]:
plot_confusion_matrix(targets, preds, labels, partition)

ValueError: At least one label specified must be in y_true